# Release Registry and Lineage

> **The story:** In 2018, Matei Zaharia and the Databricks team introduced MLflow to make machine-learning runs reproducible across tools. Model registries later made versioned model assets easier to govern, but production AI applications widened the unit of change: a model now ships with an adapter, tokenizer, prompt, retrieval index, evaluators, runtime, and rollback state. Riverside House has reached that wider boundary.
>
> **Where you are:** Fine-tuning produced pinned training manifests; prompt and retrieval work produced versioned behavior inputs; evaluation produced release evidence. What you do not yet have is one queryable release identity that binds those parts and proves they belong together. This notebook receives the shared synthetic release fixtures and delivers a read-only local registry plus explicit compatibility decisions. It writes nothing.
>
> **Notation:** $r$ is a release; $A(r)$ is its artifact set; $L(r)$ is its lineage set; $G_i(r)$ is gate $i$; $R(r)$ is its rollback target; $D(x)$ is a SHA-256 digest; $\land$ means every condition must hold.

**Inter-artifact contract**

- **What you finished earlier:** fine-tuning, prompt, retrieval, and evaluation work produced separate artifacts and evidence.
- **What this notebook delivers:** an in-memory registry that resolves exact release IDs, blocks incompatible candidates, and traces rollback lineage.
- **What the capstone still needs:** a versioned application-release manifest plus retained deployment, approval, and cloud validation evidence.

Kernels do not share memory. This notebook reloads every source from disk and treats the shared fixture directory as immutable.

## 0 - The Challenge

> **The mission:** Riverside House - identify the exact release behind every request, reject every incompatible candidate, and preserve one accepted rollback target.

**What we know so far:**

- The shared registry fixture contains 3 release records: 2 accepted and 1 blocked.
- Every record names a base, adapter, dataset, prompt, index, evaluator report, gate results, and rollback field.
- **But an artifact directory can exist while the release is unknown, and a passing evaluator can still accompany an incompatible adapter.**

**What's blocking us:**

Riverside can see model files on disk, but file presence does not answer which immutable release served a request. A `latest` alias can move. One candidate's adapter names the 360M base while its manifest supplies the 135M base. Missing prompt, index, evaluator, or rollback lineage makes reconstruction impossible.

**What this chapter unlocks:**

A release lookup returns one complete teaching manifest, a compatibility matrix, an allow/block decision, and a known accepted rollback target.

```mermaid
flowchart LR
    A["Artifact directory exists"] --> B["Failure: release unknown"]
    B --> C["Immutable release ID"]
    C --> D["Failure: parts may disagree"]
    D --> E["Schema + semantic gates"]
    E --> F["Failure: rollback unresolved"]
    F --> G["Accepted release graph"]
    style A fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style B fill:#b91c1c,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style C fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style D fill:#b91c1c,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style E fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style F fill:#b91c1c,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style G fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
```

### Topic-space ledger

| Sub-topic | Coverage | Why |
| --- | --- | --- |
| Shared manifest and schema loading | Built | This is the portable local evidence contract |
| Immutable identity and alias failure | Built | Reproduction fails without it |
| Base/adapter, gate, lineage, and rollback checks | Built | These decide whether a release may serve |
| Request-to-release lookup | Built | This answers the operational lineage question |
| Digest verification and tokenizer/runtime binding | Explained | The platform verifier owns the production implementation |
| Azure ML and Microsoft Foundry resource mapping | Explained | Cloud resources are not available or required here |
| Cloud registration, deployment, and rollback rehearsal | Named only | Those require identity, resources, approvals, and retained cloud output |

**Predict:** If all files exist and an evaluator passed, is promotion safe? Choose: (A) yes, (B) if most checks pass, or (C) only if every required structural and semantic check passes.

In [ ]:
# -- Locate and load read-only inputs ----------------------------------------
from __future__ import annotations

import copy  # pyright: ignore[reportUnusedImport]
from hashlib import sha256
import json
import re  # pyright: ignore[reportUnusedImport]
from dataclasses import dataclass  # pyright: ignore[reportUnusedImport]
from pathlib import Path
from typing import Any  # pyright: ignore[reportUnusedImport]

from jsonschema import Draft202012Validator, FormatChecker


def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "AUTHORING_GUIDE.md").is_file() and (candidate / "learning/ai-engineer/shared").is_dir():
            return candidate
    searched = " -> ".join(str(candidate) for candidate in (start, *start.parents))
    raise FileNotFoundError(
        f"Could not locate the ai-portfolio repository root from {start}. "
        "Expected both AUTHORING_GUIDE.md and learning/ai-engineer/shared in one ancestor. "
        f"Searched: {searched}"
    )


REPO_ROOT = find_repo_root(Path.cwd().resolve())
SHARED_ROOT = REPO_ROOT / "learning" / "ai-engineer" / "shared"
SHARED_DIR = SHARED_ROOT / "release-lineage"
MANIFEST_PATH = SHARED_DIR / "release-manifests.json"
SCHEMA_PATH = SHARED_DIR / "release-manifests.schema.json"
TRAINING_MANIFEST_PATH = REPO_ROOT / "checkpoints" / "instruction-lora" / "experiment-manifest.json"
PLATFORM_SCHEMA_PATH = REPO_ROOT / "projects" / "riverside-ai-platform" / "contracts" / "v1" / "model-release-manifest.schema.json"
TRACE_PATH = SHARED_ROOT / "latency-cost" / "request-traces.jsonl"

for required_path in (MANIFEST_PATH, SCHEMA_PATH, TRAINING_MANIFEST_PATH, PLATFORM_SCHEMA_PATH, TRACE_PATH):
    if not required_path.is_file():
        raise FileNotFoundError(required_path)

fixture_version = (SHARED_ROOT / "VERSION").read_text(encoding="utf-8").strip()
fixture_manifest = json.loads((SHARED_ROOT / "fixture-manifest.json").read_text(encoding="utf-8"))
if fixture_manifest["fixture_version"] != fixture_version:
    raise RuntimeError("Fixture VERSION and fixture-manifest.json disagree")
for relative_path in (
    "release-lineage/release-manifests.json",
    "release-lineage/release-manifests.schema.json",
    "release-lineage/EXPECTED_OUTCOMES.md",
    "latency-cost/request-traces.jsonl",
):
    expected_digest = fixture_manifest["files"].get(relative_path)
    if expected_digest is None:
        raise RuntimeError(f"Fixture manifest does not pin {relative_path}")
    actual_digest = sha256((SHARED_ROOT / relative_path).read_bytes()).hexdigest()
    if actual_digest != expected_digest:
        raise RuntimeError(
            f"Stale or modified fixture: {relative_path}. "
            "Restore the pinned bytes or intentionally version the shared fixture contract."
        )

manifest_document = json.loads(MANIFEST_PATH.read_text(encoding="utf-8"))
manifest_schema = json.loads(SCHEMA_PATH.read_text(encoding="utf-8"))
Draft202012Validator.check_schema(manifest_schema)
schema_validator = Draft202012Validator(manifest_schema, format_checker=FormatChecker())
schema_errors = sorted(schema_validator.iter_errors(manifest_document), key=lambda error: list(error.path))
if schema_errors:
    raise ValueError("Shared fixture failed its owning schema: " + schema_errors[0].message)

manifests = manifest_document["manifests"]
manifests_by_id = {manifest["release_id"]: manifest for manifest in manifests}
status_counts = {status: sum(item["status"] == status for item in manifests) for status in ("accepted", "blocked")}

print(f"Verified fixture contract: {fixture_version}")
print(f"Loaded {len(manifests)} schema-valid release manifests")
print(f"Accepted: {status_counts['accepted']}; blocked: {status_counts['blocked']}")
print("Answer: C - promotion is conjunction, so every required check must pass")

### Common Pitfalls

| | Pattern | Why it matters |
| --- | --- | --- |
| Wrong | Start from the notebook's current directory | Kernels may launch from the repository root or another folder |
| Right | Walk upward to a stable repository marker | Every input resolves independently of launch location |
| Wrong | Rewrite a shared fixture to make an exercise pass | Other chapters lose their stable contract |
| Right | Deep-copy a record for in-memory failure probes | The source remains unchanged and reusable |

**Quick Health Check**

Verify the schema itself is valid, the document has exactly three unique release IDs, the status split is two accepted and one blocked, and every source remains a readable file.

In [ ]:
# -- Check fixture health ---------------------------------------------------
release_ids = [manifest["release_id"] for manifest in manifests]
assert len(release_ids) == 3
assert len(set(release_ids)) == len(release_ids)
assert status_counts == {"accepted": 2, "blocked": 1}
assert MANIFEST_PATH.stat().st_size > 0 and SCHEMA_PATH.stat().st_size > 0
print("PASS: shared fixture shape, IDs, statuses, and files match the chapter contract")

## 1 - A Directory Is Not a Release

An artifact is bytes plus identity. A registered artifact adds an immutable locator and metadata. A release binds compatible artifacts to evidence and an operational decision. A directory gives you only a location.

```mermaid
flowchart LR
    A["Directory\npath exists"] --> B["Artifact\nbytes + digest"]
    B --> C["Registered artifact\nimmutable ID + version"]
    C --> D["Application release\nlineage + gates + rollback"]
    A -."does not imply".-> D
    style A fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style B fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style C fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style D fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
```

Let $P$ mean "the artifact directory exists" and $K$ mean "the release is known." One observation with $P = \text{true}$ and $K = \text{false}$ disproves $P \Rightarrow K$. File presence cannot be your promotion predicate.

**Predict:** The checkpoint directory below exists. Does that prove `rel-riv-999` is registered?

In [ ]:
# -- Prove artifact presence does not identify a release --------------------
artifact_directory = REPO_ROOT / "checkpoints" / "instruction-lora"
unknown_release_id = "rel-riv-999"
directory_exists = artifact_directory.is_dir()
release_known = unknown_release_id in manifests_by_id
assert directory_exists and not release_known

training_manifest = json.loads(TRAINING_MANIFEST_PATH.read_text(encoding="utf-8"))
training_fields = {"model", "stage", "seed", "training_files", "evaluation_files"}
release_only_fields = {"release_id", "prompt", "index", "evaluators", "rollback_target"}
assert training_fields.issubset(training_manifest)
assert release_only_fields.isdisjoint(training_manifest)

print(f"Artifact directory exists: {directory_exists}")
print(f"Release {unknown_release_id} known: {release_known}")
print("PASS: training provenance is rich, but neither a directory nor training manifest is an application release")

**Your turn:** Change the proposed release ID to one that exists. The result should move from `UNKNOWN` to `KNOWN` without inspecting the artifact directory.

### Common Pitfalls

| | Pattern | Why it matters |
| --- | --- | --- |
| Wrong | Promote when `artifact_dir.exists()` | Partial, stale, or unregistered bytes can receive traffic |
| Right | Resolve an immutable release, then verify every referenced artifact | Identity and integrity become separate explicit checks |
| Wrong | Treat a training manifest as a serving release | Prompt, index, evaluation, runtime, and rollback state disappear |
| Right | Reference unchanged training provenance from a release contract | Training evidence remains auditable without being rewritten |

**Quick Health Check**

Ask: Is the release ID known? Is every artifact reference immutable? Does every digest verify? Is there an explicit release decision? This notebook builds identity and decision checks; the platform verifier owns byte-digest verification.

In [ ]:
# -- Exercise: separate path presence from registry identity ----------------
# CHANGE THIS: replace rel-riv-999 with an exact ID from the loaded fixture.
proposed_release_id = "rel-riv-999"
lookup_state = "KNOWN" if proposed_release_id in manifests_by_id else "UNKNOWN"
print(f"{proposed_release_id}: {lookup_state}")
print("PASS when the result is KNOWN because the registry, not the path, supplied identity")

## 2 - Mutable Aliases Break Reproduction

Aliases are useful routing controls. They are poor evidence. The same name can point to different releases at different times:

$$\operatorname{resolve}(\text{latest}, t_1) \ne \operatorname{resolve}(\text{latest}, t_2).$$

The lookup depends on time. A historical request carrying only `latest` cannot tell you which bytes, prompt, or index answered it.

```mermaid
flowchart TB
    A["Client model alias\nriverside-editor"] --> B["Traffic policy"]
    B -->|"time t1"| C["rel-riv-001"]
    B -->|"time t2"| D["rel-riv-002"]
    E["Request evidence"] --> F["Exact release_id\nrel-riv-002"]
    style A fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style B fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style C fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style D fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style E fill:#b91c1c,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style F fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
```

A stable client-facing alias is allowed at the API boundary. The response and telemetry must still record the immutable release ID that actually served.

**Predict:** If `latest` points to two IDs over time, can the alias alone reproduce a request?

In [ ]:
# -- Prove the mutable-alias failure and exercise the fix -------------------
alias_snapshots = {
    "2026-01-10T12:00:00Z": {"latest": "rel-riv-001"},
    "2026-02-01T12:00:00Z": {"latest": "rel-riv-002"},
}
resolved_ids = [snapshot["latest"] for snapshot in alias_snapshots.values()]
assert len(set(resolved_ids)) == 2
IMMUTABLE_RELEASE_ID = re.compile(r"^rel-riv-[0-9]{3}$")

# CHANGE THIS: replace latest with the exact release that served.
recorded_identity = "latest"
identity_is_immutable = IMMUTABLE_RELEASE_ID.fullmatch(recorded_identity) is not None

print(f"The alias resolved to: {resolved_ids}")
print(f"Immutable request identity: {identity_is_immutable}")
print("PASS when identity is True: routing aliases and release evidence now have separate jobs")

### Common Pitfalls

| | Pattern | Why it matters |
| --- | --- | --- |
| Wrong | Store `latest` in a trace | Replaying the trace may select different artifacts |
| Right | Store exact `release_id`; keep alias separately as routing context | Historical identity survives traffic changes |
| Wrong | Ban all aliases | Clients become coupled to physical deployments |
| Right | Keep stable API aliases while emitting exact release/deployment metadata | Routing flexibility and auditability coexist |

**Quick Health Check**

Verify release IDs match the immutable namespace, traces never use `latest`, and alias snapshots are retained with traffic changes.

In [ ]:
# -- Check immutable request identity ---------------------------------------
assert all(IMMUTABLE_RELEASE_ID.fullmatch(release_id) for release_id in manifests_by_id)
assert IMMUTABLE_RELEASE_ID.fullmatch("latest") is None
assert set(resolved_ids).issubset(manifests_by_id)
print("PASS: registry IDs are immutable; the moving alias remains separate routing state")

## 3 - Schema Valid Is Not Compatible

JSON Schema can require fields, types, patterns, and minimum lengths. Portable schema cannot generally compare two sibling values. That leaves a semantic gap: `rel-riv-003` satisfies the shared schema while its adapter declares a different compatible base model.

```mermaid
flowchart LR
    A["JSON document"] --> B{"Schema valid?"}
    B -->|"no"| C["Block: malformed"]
    B -->|"yes"| D{"Semantic invariants?"}
    D -->|"no"| E["Block: incompatible"]
    D -->|"yes"| F["Eligible for remaining gates"]
    style A fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style B fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style C fill:#b91c1c,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style D fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style E fill:#b91c1c,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style F fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
```

The required equality is:

$$\text{adapter.compatible\_base\_model\_id} = \text{base\_model.artifact\_id}.$$

The adapter's declared parent must be the exact base supplied by the release. Similar names, families, or parameter counts are not enough.

**Predict:** Which omission is harmless: prompt, index, or evaluator? None; each changes behavior or the evidence used to approve it.

In [ ]:
# -- Prove schema acceptance does not imply compatibility ------------------
incompatible = manifests_by_id["rel-riv-003"]
single_record_document = {
    "schema_version": manifest_document["schema_version"],
    "manifests": [incompatible],
}
schema_accepts = not list(schema_validator.iter_errors(single_record_document))
base_adapter_matches = (
    incompatible["adapter"]["compatible_base_model_id"]
    == incompatible["base_model"]["artifact_id"]
)
assert schema_accepts and not base_adapter_matches

missing_lineage_results: dict[str, list[str]] = {}
for missing_field in ("prompt", "index", "evaluators"):
    damaged_release = copy.deepcopy(manifests_by_id["rel-riv-002"])
    damaged_release.pop(missing_field)
    damaged_document = {
        "schema_version": manifest_document["schema_version"],
        "manifests": [damaged_release],
    }
    messages = [error.message for error in schema_validator.iter_errors(damaged_document)]
    missing_lineage_results[missing_field] = messages
    print(f"Missing {missing_field}: BLOCK ({messages[0]})")

assert all(missing_lineage_results.values())
print(f"Schema accepts rel-riv-003: {schema_accepts}")
print(f"Base/adapter compatible: {base_adapter_matches}")
print("PASS: semantic equality and complete lineage block what shape alone cannot")

### Common Pitfalls

| | Pattern | Why it matters |
| --- | --- | --- |
| Wrong | Stop after JSON Schema passes | Sibling equality and graph invariants remain unchecked |
| Right | Run schema first, then named semantic checks | Failures are precise and ordered |
| Wrong | Accept a passing evaluator as an override | Evaluation cannot make incompatible artifacts load correctly |
| Right | Combine required gates with logical AND | One hard failure blocks promotion |
| Wrong | Fill missing lineage from current defaults | Historical behavior becomes unreproducible |
| Right | Require exact prompt, index, and evaluator IDs | Reconstruction is independent of today's configuration |

**Quick Health Check**

Validate shape, compare base/adapter identity, require every gate and evaluator to pass, and refuse missing behavior lineage.

In [ ]:
# -- Check compatibility facts for every shared release --------------------
for manifest in manifests:
    checks = {
        "base_adapter": manifest["adapter"]["compatible_base_model_id"] == manifest["base_model"]["artifact_id"],
        "all_gates": all(manifest["gate_results"].values()),
        "all_evaluators": bool(manifest["evaluators"]) and all(item["passed"] for item in manifest["evaluators"]),
        "lineage_present": all(field in manifest for field in ("prompt", "index", "evaluators")),
    }
    print(manifest["release_id"], checks)

assert all(manifests_by_id["rel-riv-002"]["gate_results"].values())
assert not all(manifests_by_id["rel-riv-003"]["gate_results"].values())
print("PASS: rel-riv-003 remains blocked despite its passing evaluator report")

## 4 - Rollback Is a Graph, Not a Label

A rollback target is useful only if it resolves to a known, accepted, older release. A string can be present and still be dangling, self-referential, blocked, or newer than the candidate.

```mermaid
flowchart RL
    B["rel-riv-002\naccepted"] -->|"rollback"| A["rel-riv-001\naccepted initial"]
    C["rel-riv-003\nblocked"] -."never active".-> B
    D["rel-riv-999\nmissing"] -."dangling target".-> B
    style A fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style B fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style C fill:#b91c1c,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style D fill:#b91c1c,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
```

For every non-initial accepted release:

$$R(r) \in \mathcal{A} \land t(R(r)) < t(r),$$

where $\mathcal{A}$ is the accepted set. The target must exist inside that set and predate the candidate.

In [ ]:
# -- Validate rollback graph and produce failure probes --------------------
def rollback_issues(manifest: dict[str, Any], records: dict[str, dict[str, Any]]) -> list[str]:
    accepted = sorted(
        (item for item in records.values() if item["status"] == "accepted"),
        key=lambda item: item["created_at_utc"],
    )
    if manifest["status"] != "accepted":
        return []
    if manifest["release_id"] == accepted[0]["release_id"]:
        return [] if manifest["rollback_target"] is None else ["initial release must not invent a predecessor"]

    target_id = manifest["rollback_target"]
    if target_id is None:
        return ["non-initial accepted release has no rollback target"]
    if target_id == manifest["release_id"]:
        return ["release cannot roll back to itself"]
    target = records.get(target_id)
    if target is None:
        return ["rollback target is unknown"]
    if target["status"] != "accepted":
        return ["rollback target is not accepted"]
    if target["created_at_utc"] >= manifest["created_at_utc"]:
        return ["rollback target does not predate release"]
    return []


for release_id in ("rel-riv-001", "rel-riv-002"):
    assert not rollback_issues(manifests_by_id[release_id], manifests_by_id)

for probe_name, target_id in {"absent": None, "dangling": "rel-riv-999", "self": "rel-riv-002"}.items():
    damaged = copy.deepcopy(manifests_by_id["rel-riv-002"])
    damaged["rollback_target"] = target_id
    issues = rollback_issues(damaged, manifests_by_id)
    assert issues
    print(f"{probe_name}: BLOCK ({issues[0]})")

print("PASS: valid accepted edges resolve; absent, dangling, and self targets fail closed")

**Your turn:** Point the copied candidate at `rel-riv-001`. The result should be an empty issue list.

### Common Pitfalls

| | Pattern | Why it matters |
| --- | --- | --- |
| Wrong | Store `previous` or `last-known-good` | The target can move or become ambiguous |
| Right | Store the exact accepted release ID | The rollback edge is reproducible |
| Wrong | Check only that the target string is non-empty | Dangling, self, blocked, and future edges survive |
| Right | Resolve status and chronology in the release graph | The target is demonstrably prior accepted state |
| Wrong | Declare success after changing traffic | Index, policy, citations, and health may still be wrong |
| Right | Retain timed end-to-end rehearsal evidence | Operational recovery, not command success, is proved |

**Quick Health Check**

Resolve the target, require accepted status, require an earlier timestamp, reject cycles, and rehearse model plus index restoration in staging before claiming production rollback.

In [ ]:
# -- Exercise: repair the rollback edge ------------------------------------
candidate = copy.deepcopy(manifests_by_id["rel-riv-002"])
# CHANGE THIS: set the exact accepted predecessor.
candidate["rollback_target"] = "rel-riv-001"
candidate_rollback_issues = rollback_issues(candidate, manifests_by_id)
rollback_exercise_passed = candidate_rollback_issues == []
print(f"{'PASS' if rollback_exercise_passed else 'FAIL'}: rollback target {candidate['rollback_target']} resolves to an accepted predecessor")
print("Takeaway: a rollback label is useful only after graph resolution and chronology checks.")

## 5 - Build the Local Release Registry

You now have the pieces that make a registry useful: exact lookup, duplicate rejection, compatibility checks, complete lineage resolution, request attribution, and rollback traversal. The implementation stays small enough to inspect top to bottom.

```mermaid
flowchart LR
    A["Shared manifest document"] --> B["Schema validator"]
    B --> C["Unique ID index"]
    C --> D["Compatibility checks"]
    D --> E{"All checks true?"}
    E -->|"yes"| F["Resolve lineage + rollback"]
    E -->|"no"| G["Block with reasons"]
    style A fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style B fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style C fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style D fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style E fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style F fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style G fill:#b91c1c,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
```

The registry does not guess defaults and does not resolve aliases. A caller must supply the exact release ID recorded in request or deployment metadata.

In [ ]:
# -- Implement an inspectable local registry -------------------------------
@dataclass(frozen=True)
class ReleaseDecision:
    release_id: str
    promotable: bool
    checks: dict[str, bool]
    reasons: tuple[str, ...]


class LocalReleaseRegistry:
    def __init__(self, document: dict[str, Any], validator: Draft202012Validator) -> None:
        errors = sorted(validator.iter_errors(document), key=lambda error: list(error.path))
        if errors:
            raise ValueError(f"release document is schema-invalid: {errors[0].message}")
        records = copy.deepcopy(document["manifests"])
        release_ids = [record["release_id"] for record in records]
        if len(release_ids) != len(set(release_ids)):
            raise ValueError("release IDs must be unique")
        self._records = {record["release_id"]: record for record in records}

    def get(self, release_id: str) -> dict[str, Any]:
        if IMMUTABLE_RELEASE_ID.fullmatch(release_id) is None:
            raise KeyError(f"exact immutable release ID required: {release_id}")
        if release_id not in self._records:
            raise KeyError(f"unknown release: {release_id}")
        return copy.deepcopy(self._records[release_id])

    def checks(self, release_id: str) -> dict[str, bool]:
        manifest = self.get(release_id)
        return {
            "immutable_identity": IMMUTABLE_RELEASE_ID.fullmatch(release_id) is not None,
            "accepted_status": manifest["status"] == "accepted",
            "base_adapter_compatibility": manifest["adapter"]["compatible_base_model_id"] == manifest["base_model"]["artifact_id"],
            "all_gate_results": all(manifest["gate_results"].values()),
            "all_evaluators_passed": bool(manifest["evaluators"]) and all(item["passed"] for item in manifest["evaluators"]),
            "complete_behavior_lineage": all(field in manifest for field in ("prompt", "index", "evaluators")),
            "rollback_resolves": not rollback_issues(manifest, self._records),
        }

    def decide(self, release_id: str) -> ReleaseDecision:
        manifest = self.get(release_id)
        checks = self.checks(release_id)
        reasons = [name.replace("_", " ") for name, passed in checks.items() if not passed]
        reasons.extend(manifest["blocked_reasons"] if manifest["status"] == "blocked" else [])
        return ReleaseDecision(release_id, all(checks.values()), checks, tuple(dict.fromkeys(reasons)))

    def lineage(self, release_id: str) -> dict[str, Any]:
        decision = self.decide(release_id)
        if not decision.promotable:
            raise ValueError(f"release cannot serve: {decision.reasons}")
        manifest = self.get(release_id)
        fields = ("release_id", "base_model", "adapter", "dataset", "prompt", "index", "evaluators", "rollback_target")
        return {field: copy.deepcopy(manifest[field]) for field in fields}

    def resolve_request(self, request: dict[str, Any]) -> dict[str, Any]:
        if "release_id" not in request:
            raise KeyError("request has no release_id lineage")
        return self.lineage(request["release_id"])


registry = LocalReleaseRegistry(manifest_document, schema_validator)
print("PASS: registry loaded a schema-valid document with unique immutable IDs")

### Code Walkthrough: `LocalReleaseRegistry`

1. **Constructor ordering** - schema validation runs before indexing, so malformed records never enter. Duplicate detection follows because JSON Schema does not enforce uniqueness by `release_id`.
2. **`get` exactness** - aliases and malformed identifiers fail before lookup. Returning a deep copy prevents accidental history mutation.
3. **`checks` composition** - each invariant has a stable name and boolean result, so failures remain observable.
4. **`decide` conjunction** - `all(checks.values())` is the promotion rule. Stored blocked reasons add context; they never replace computed checks.
5. **`lineage` fail-closed behavior** - blocked releases cannot be resolved as serving candidates, even when operators can inspect their records.
6. **`resolve_request` boundary** - attribution starts from the exact `release_id`; no current alias or deployment default is consulted.

In [ ]:
# -- Prove registry decisions and request lineage --------------------------
expected_decisions = {
    "rel-riv-001": True,
    "rel-riv-002": True,
    "rel-riv-003": False,
}
for release_id, expected in expected_decisions.items():
    decision = registry.decide(release_id)
    assert decision.promotable is expected
    print(f"{release_id}: {'ALLOW' if decision.promotable else 'BLOCK'} {decision.reasons}")

resolved = registry.lineage("rel-riv-002")
assert resolved["rollback_target"] == "rel-riv-001"
assert resolved["adapter"]["artifact_id"] == "adapter-riv-sft-002"

request_traces = [json.loads(line) for line in TRACE_PATH.read_text(encoding="utf-8").splitlines() if line.strip()]
assert {trace["release_id"] for trace in request_traces} == {"rel-riv-002"}
for trace in request_traces:
    assert registry.resolve_request(trace)["release_id"] == trace["release_id"]

print(f"PASS: resolved {len(request_traces)} requests to exact rel-riv-002 lineage; rel-riv-003 cannot serve")

### Common Pitfalls

| | Pattern | Why it matters |
| --- | --- | --- |
| Wrong | Index records and ignore duplicate IDs | Later records silently overwrite evidence |
| Right | Reject duplicates before constructing the index | Identity remains one-to-one |
| Wrong | Return mutable registry records | A caller can alter apparent history |
| Right | Return immutable views or defensive copies | Registry state remains controlled |
| Wrong | Return blocked lineage to serving code | A readable record becomes an accidental candidate |
| Right | Separate audit lookup from serve-eligible resolution | Operators can inspect failures without routing traffic |

**Quick Health Check**

Test duplicate rejection, unknown lookup, alias rejection, expected allow/block decisions, complete lineage, request attribution, and the immediate rollback target.

In [ ]:
# -- Check local registry failure boundaries -------------------------------
duplicate_document = copy.deepcopy(manifest_document)
duplicate_document["manifests"].append(copy.deepcopy(duplicate_document["manifests"][0]))

failure_probes: dict[str, bool] = {}
for probe_name, probe in {
    "duplicate": lambda: LocalReleaseRegistry(duplicate_document, schema_validator),
    "unknown": lambda: registry.get("rel-riv-999"),
    "alias": lambda: registry.get("latest"),
    "blocked_lineage": lambda: registry.lineage("rel-riv-003"),
    "missing_request_lineage": lambda: registry.resolve_request({"request_id": "req-missing"}),
}.items():
    try:
        probe()
        failure_probes[probe_name] = False
    except (KeyError, ValueError):
        failure_probes[probe_name] = True

assert all(failure_probes.values())
print(f"PASS: every registry failure probe blocked as expected: {failure_probes}")

**Reflection bridge:** The local registry can reconstruct and reject fixture releases, but it has not registered an Azure asset, inspected a live endpoint, or rehearsed rollback. The next step is to map each portable contract field to a cloud evidence owner without relabeling source inspection as deployment proof.

## 6 - Map the Portable Contract to Azure Evidence

The local manifest remains the join key. Azure ML can own model artifacts and endpoint deployments; Microsoft Foundry can own project/evaluation assets; prompt and index metadata may live in separate governed stores. No single service automatically proves the complete application release.

```mermaid
flowchart LR
    A["Portable release manifest"] --> B["Azure ML model/version"]
    A --> C["Endpoint deployment metadata"]
    A --> D["Foundry evaluation/project assets"]
    A --> E["Prompt + index metadata"]
    B --> F{"Live IDs and digests retained?"}
    C --> F
    D --> F
    E --> F
    F -->|no| G["UNVALIDATED cloud claim"]
    F -->|yes| H["Release-scoped cloud evidence"]
    style A fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style B fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style C fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style D fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style E fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style F fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style G fill:#b91c1c,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style H fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
```

| Release field | Likely Azure owner | Evidence required before claiming live validation |
|---|---|---|
| Base model and adapter | Azure ML registry/model assets | Immutable resource IDs, versions, digests, compatibility verification |
| Runtime and deployment slot | Azure ML managed online endpoint | Deployment ID, image/environment digest, traffic state, readiness evidence |
| Evaluation report | Microsoft Foundry or retained evaluation artifact | Dataset/evaluator versions, release binding, thresholds, retained results |
| Prompt and retrieval index | Governed prompt/index metadata stores | Immutable IDs, content/config digests, authorization and freshness evidence |
| Rollback target | Deployment control plane plus release graph | Accepted target, rehearsed model/index/policy restoration, elapsed recovery time |

**Predict:** Does reading valid local schemas justify the label `LIVE_AZURE`, or only `IMPLEMENTED_SOURCE` plus local fixture evidence?

In [ ]:
# ── Check the local-to-cloud evidence boundary ──────────────────────────
platform_schema = json.loads(PLATFORM_SCHEMA_PATH.read_text(encoding="utf-8"))
Draft202012Validator.check_schema(platform_schema)
cloud_evidence = {
    "portable_fixture_loaded": True,
    "platform_contract_inspected": True,
    "azure_registry_write_observed": False,
    "endpoint_deployment_observed": False,
    "rollback_rehearsal_observed": False,
}
evidence_labels = {
    "manifest_contract": "IMPLEMENTED_SOURCE",
    "local_registry_mechanics": "LOCAL_FIXTURE when executed; UNVALIDATED in authored state",
    "cloud_registration": "UNVALIDATED",
    "cloud_deployment": "UNVALIDATED",
    "cloud_rollback": "UNVALIDATED",
}
assert cloud_evidence["portable_fixture_loaded"] and cloud_evidence["platform_contract_inspected"]
assert not any(value for key, value in cloud_evidence.items() if key.endswith("observed"))
print(json.dumps(evidence_labels, indent=2))
print("Prediction resolved: local source and fixture checks do not become LIVE_AZURE evidence.")

**Common Pitfalls**

| | Pattern | Why it matters |
|---|---|---|
| Wrong | Call a schema file an Azure deployment | Source presence proves no control-plane action |
| Wrong | Treat a model registry version as the whole application release | Prompt, index, evaluator, runtime, and rollback can still differ |
| Right | Retain immutable cloud resource IDs beside the portable manifest | Local and cloud evidence can be joined without guessing |

**Quick Health Check:** every cloud claim names a subscription/project, region, environment, release, workload, time window, retained output, and reviewer; absent fields force `UNVALIDATED`.

In [ ]:
# ── Health check: reject inflated cloud evidence labels ─────────────────
required_live_fields = {
    "subscription_or_project", "region", "environment", "release_id",
    "workload", "time_window", "retained_output", "reviewer",
}
authored_cloud_record = {"release_id": "rel-riv-002", "environment": "not contacted"}
missing_live_fields = sorted(required_live_fields - authored_cloud_record.keys())
cloud_label = "LIVE_AZURE" if not missing_live_fields else "UNVALIDATED"
assert cloud_label == "UNVALIDATED"
print(f"PASS: cloud label={cloud_label}; missing required evidence fields={missing_live_fields}")
print("Takeaway: a service mapping is architecture, not retained operational evidence.")

**Your turn:** Add every required field to a copied cloud record only after an authorized run has retained real values. The correctness check must remain `UNVALIDATED` for placeholders.

In [ ]:
# ── Your turn: test evidence-label completeness without inventing proof ─
MY_CLOUD_RECORD = {  # CHANGE THIS only with retained, authorized evidence
    "release_id": "rel-riv-002",
    "environment": "UNVALIDATED_PLACEHOLDER",
}
my_missing_fields = sorted(required_live_fields - MY_CLOUD_RECORD.keys())
contains_placeholder = any("PLACEHOLDER" in str(value) for value in MY_CLOUD_RECORD.values())
my_label = "LIVE_AZURE" if not my_missing_fields and not contains_placeholder else "UNVALIDATED"
print(f"Evidence label: {my_label}")
print(f"Missing fields: {my_missing_fields}")
print(f"Correctness check: {'PASS' if my_label == 'UNVALIDATED' else 'REVIEW REQUIRED'}")

**Reflection bridge:** The registry can now separate what source, a local fixture, and a live cloud run can each prove. Riverside still needs one compact handoff that states which release may serve, which one must not, and where rollback points.

## 7 - Package the Release Decision and Hand Off

```mermaid
flowchart LR
    A["Schema + semantic checks"] --> B["Registry decisions"]
    B --> C["Request lineage"]
    B --> D["Rollback graph"]
    C --> E["Portable release report"]
    D --> E
    E --> F["Capstone evidence package"]
    style A fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style B fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style C fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style D fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style E fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style F fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
```

In [ ]:
# ── Closing scorecard: assemble release and lineage evidence ────────────
release_scorecard = []
for release_id in sorted(expected_decisions):
    decision = registry.decide(release_id)
    release_scorecard.append({
        "release_id": release_id,
        "expected_promotable": expected_decisions[release_id],
        "observed_promotable": decision.promotable,
        "failed_checks": [name for name, passed in decision.checks.items() if not passed],
        "rollback_target": manifests_by_id[release_id]["rollback_target"],
    })
assert all(row["expected_promotable"] == row["observed_promotable"] for row in release_scorecard)
release_report = {
    "schema_version": "ai-eng.release-lineage-report.v1",
    "evidence_label": "LOCAL_FIXTURE when executed; UNVALIDATED in authored state",
    "request_release_ids": sorted({trace["release_id"] for trace in request_traces}),
    "decisions": release_scorecard,
    "serving_release": "rel-riv-002",
    "rollback_target": resolved["rollback_target"],
    "blocked_release": "rel-riv-003",
    "cloud_validation": "UNVALIDATED",
}
print(json.dumps(release_report, indent=2))
print("Riverside decision: rel-riv-002 is the accepted fixture release; rel-riv-003 remains blocked.")
print("Takeaway: promotion is a conjunction; evaluator success cannot compensate for incompatibility.")

### Three-Tier Coverage Ledger

| Tier | Techniques | Evidence boundary |
|---|---|---|
| Built and measured | Shared schema validation; immutable release lookup; alias counterexample; base/adapter compatibility; complete prompt/index/evaluator lineage; AND promotion gates; rollback graph; duplicate protection; defensive-copy registry; request attribution; failure probes | Deterministic fixture workflow executed successfully in the unified FDE environment; notebook outputs were cleared |
| Explained and illustrated | Platform model-release schema; Azure ML model/deployment mapping; Foundry evaluation/project mapping; prompt/index metadata ownership; cloud evidence completeness | Local source mapping only; no Azure control-plane action occurred |
| Named with a reason | Registry upload; endpoint deployment; traffic switch; managed-identity authorization; digest download verification; timed rollback rehearsal | These require authorized cloud resources, approvals, workload, and retained live output |

If you find a technique named above that does not appear in the tier table, that is exactly the bug this section exists to catch.

### Completed Roadmap

| Step | Failure exposed | What the notebook now proves when run |
|---:|---|---|
| 1 | A directory was mistaken for a release | Existing bytes do not make `rel-riv-999` known |
| 2 | `latest` moved over time | Immutable release IDs reproduce request identity |
| 3 | Schema validity hid incompatibility | `rel-riv-003` is blocked despite a passing evaluator |
| 4 | Rollback was treated as a label | `rel-riv-002` resolves to earlier accepted `rel-riv-001` |
| 5 | Records lacked a queryable boundary | The local registry rejects duplicates, aliases, unknowns, and blocked lineage |
| 6 | Architecture mapping was inflated into cloud proof | Azure claims remain `UNVALIDATED` without retained live fields |
| 7 | Evidence was scattered | One report names serving, blocked, and rollback releases |

### Key Takeaways

- A model artifact is one component, not an application release.
- Mutable aliases route traffic; immutable IDs explain history.
- Schema validates shape; semantic checks validate relationships.
- Promotion is an AND across non-compensating gates.
- Rollback is a validated graph edge plus a rehearsed operation.
- Local fixture evidence and Azure evidence require different labels.

> **Next:** [Production Feedback and Drift](../05-production-feedback-and-drift/README.md) uses exact release lineage to turn incidents into reviewed evaluation candidates. The [capstone](../06-capstone/README.md) consumes this release report as one gate in the final Riverside decision.